# Product Demand Prediction
## Business problem
Can we predict the demand for a product using its price, discount, rating, category, brand, inventory, and historical sales behavior?

In [1]:
import pandas as pd
import numpy as np
customers = pd.read_csv("customers.csv")
orders = pd.read_csv("orders.csv")
products = pd.read_csv("products.csv")

print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Products:", products.shape)

Customers: (60000, 9)
Orders: (200000, 14)
Products: (10000, 8)


In [2]:
customers.head()

,customer_id,customer_name,age,gender,city,state,customer_segment,signup_date,preferred_channel
0,CUST000001,Customer_000001,27,Female,Pune,Maharashtra,Premium,2023-08-06,Website
1,CUST000002,Customer_000002,45,Male,Hyderabad,Telangana,Regular,2025-06-24,Website
2,CUST000003,Customer_000003,53,Female,Surat,Gujarat,Premium,2025-02-22,Mobile App
3,CUST000004,Customer_000004,27,Male,Bhopal,Madhya Pradesh,Regular,2023-08-17,Website
4,CUST000005,Customer_000005,31,Male,Noida,Uttar Pradesh,New,2023-02-07,Website


In [3]:
orders.head()

,order_id,customer_id,product_id,order_date,order_status,quantity,unit_price,discount_pct,discount_amount,freight_amount,total_amount,payment_method,delivery_date,estimated_delivery_date
0,ORD0000001,CUST043981,PROD02971,2024-03-04,Delivered,3,255.56,15,115.00,193.45,845.13,Debit Card,2024-03-14,2024-03-12
1,ORD0000002,CUST011019,PROD06966,2023-07-09,Delivered,1,277.63,0,0.00,391.43,669.06,Debit Card,2023-07-13,2023-07-17
2,ORD0000003,CUST007980,PROD00423,2023-02-22,Delivered,1,1207.13,15,181.07,386.09,1412.15,UPI,2023-02-25,2023-03-01
3,ORD0000004,CUST014065,PROD02332,2025-12-16,Delivered,1,256.62,0,0.00,190.21,446.83,Debit Card,2025-12-20,2025-12-23
4,ORD0000005,CUST017410,PROD02877,2024-08-27,Delivered,1,788.24,0,0.00,283.30,1071.54,UPI,2024-08-29,2024-09-03


In [4]:
products.head()

,product_id,product_name,category,sub_category,brand,unit_price,stock_quantity,rating
0,PROD00001,Product_00001,Sports,Fitness Equipment,Orion,446.04,266,4.5
1,PROD00002,Product_00002,Electronics,Laptop,Vertex,594.89,436,3.8
2,PROD00003,Product_00003,Books,Non-Fiction,UrbanX,814.08,229,4.9
3,PROD00004,Product_00004,Fashion,Jeans,PrimeTech,254.57,291,4.7
4,PROD00005,Product_00005,Home,Chair,Vertex,350.36,451,4.5


# STEP 2 — Convert date columns

In [5]:
customers['signup_date'] = pd.to_datetime(
    customers['signup_date']
)

orders['order_date'] = pd.to_datetime(
    orders['order_date']
)

orders['delivery_date'] = pd.to_datetime(
    orders['delivery_date']
)

orders['estimated_delivery_date'] = pd.to_datetime(
    orders['estimated_delivery_date']
)

# STEP 3 — Select completed sales
For demand, we don't want cancelled orders to count as actual demand.

In [6]:
delivered_orders = orders[
    orders['order_status'] == 'Delivered'
].copy()

In [7]:
delivered_orders.shape

(155934, 14)

# STEP 4 — Connect Orders with Products
Now we bring product characteristics into the sales data.

In [9]:
sales_data = delivered_orders.merge(
    products,
    on='product_id',
    how='left',
    suffixes=('_order', '_product')
)
sales_data

,order_id,customer_id,product_id,order_date,order_status,quantity,unit_price_order,discount_pct,discount_amount,freight_amount,...,payment_method,delivery_date,estimated_delivery_date,product_name,category,sub_category,brand,unit_price_product,stock_quantity,rating
0,ORD0000001,CUST043981,PROD02971,2024-03-04,Delivered,3,255.56,15,115.00,193.45,...,Debit Card,2024-03-14,2024-03-12,Product_02971,Sports,Outdoor Gear,PrimeTech,255.56,412,4.2
1,ORD0000002,CUST011019,PROD06966,2023-07-09,Delivered,1,277.63,0,0.00,391.43,...,Debit Card,2023-07-13,2023-07-17,Product_06966,Fashion,T-Shirt,Apex,277.63,389,3.1
2,ORD0000003,CUST007980,PROD00423,2023-02-22,Delivered,1,1207.13,15,181.07,386.09,...,UPI,2023-02-25,2023-03-01,Product_00423,Beauty,Haircare,Orion,1207.13,243,3.9
3,ORD0000004,CUST014065,PROD02332,2025-12-16,Delivered,1,256.62,0,0.00,190.21,...,Debit Card,2025-12-20,2025-12-23,Product_02332,Beauty,Skincare,UrbanX,256.62,396,4.0
4,ORD0000005,CUST017410,PROD02877,2024-08-27,Delivered,1,788.24,0,0.00,283.30,...,UPI,2024-08-29,2024-09-03,Product_02877,Beauty,Haircare,Nova,788.24,481,2.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155929,ORD0199994,CUST004494,PROD03171,2023-12-08,Delivered,1,1017.97,15,152.70,223.29,...,Credit Card,2023-12-12,2023-12-13,Product_03171,Electronics,Smartphone,PrimeTech,1017.97,139,2.5
155930,ORD0199995,CUST015565,PROD06859,2024-10-15,Delivered,1,961.10,10,96.11,229.51,...,Credit Card,2024-10-20,2024-10-23,Product_06859,Fashion,Jacket,Vertex,961.10,1,4.3
155931,ORD0199996,CUST005109,PROD00944,2023-09-29,Delivered,2,361.69,15,108.51,356.89,...,UPI,2023-10-02,2023-10-07,Product_00944,Sports,Fitness Equipment,Vertex,361.69,58,4.2
155932,ORD0199997,CUST031262,PROD05971,2025-03-02,Delivered,1,183.72,5,9.19,315.78,...,Debit Card,2025-03-08,2025-03-06,Product_05971,Beauty,Makeup,PrimeTech,183.72,387,3.7


Now we have information such as:
product_id
product_name
category
sub_category
brand
unit_price
stock_quantity
rating
quantity
discount_pct
total_amount
order_date

# Why do we need this?
Because we want to answer:
Does product price, discount, rating, category, etc. help explain or predict demand?

# STEP 5 — Create the product-level demand dataset
We need one row per product, rather than one row per order.

In [11]:
total_quantity = sales_data['quantity'].sum()

print("Total Quantity Sold:", total_quantity)

Total Quantity Sold: 272636


In [12]:
total_orders = sales_data['order_id'].nunique()

print("Total Delivered Orders:", total_orders)

Total Delivered Orders: 155934


In [13]:
category_demand = (
    sales_data
    .groupby('category')
    .agg(
        quantity_sold=('quantity', 'sum'),
        orders=('order_id', 'nunique')
    )
    .reset_index()
    .sort_values(
        'quantity_sold',
        ascending=False
    )
)

category_demand

,category,quantity_sold,orders
5,Sports,47605,27046
4,Home,46374,26548
3,Fashion,45555,26174
1,Books,45333,25938
2,Electronics,44197,25187
0,Beauty,43572,25041


In [15]:
product_demand = (
    sales_data
    .groupby(['product_id', 'product_name'])
    .agg(
        quantity_sold=('quantity', 'sum'),
        orders=('order_id', 'nunique')
    )
    .reset_index()
    .sort_values(
        'quantity_sold',
        ascending=False
    )
)

product_demand.head(10)

,product_id,product_name,quantity_sold,orders
5628,PROD05629,Product_05629,63,30
2274,PROD02275,Product_02275,61,25
9616,PROD09617,Product_09617,59,28
5826,PROD05827,Product_05827,59,25
7045,PROD07046,Product_07046,58,25
1416,PROD01417,Product_01417,58,27
2075,PROD02076,Product_02076,58,25
1261,PROD01262,Product_01262,57,29
3912,PROD03913,Product_03913,56,23
7406,PROD07407,Product_07407,56,24


# 5.5 Create order month

In [16]:
sales_data['order_month'] = (
    sales_data['order_date']
    .dt.to_period('M')
)

# 5.6 Monthly demand

In [17]:
monthly_demand = (
    sales_data
    .groupby('order_month')
    .agg(
        quantity_sold=('quantity', 'sum'),
        orders=('order_id', 'nunique')
    )
    .reset_index()
)
monthly_demand.head()

,order_month,quantity_sold,orders
0,2023-01,7779,4434
1,2023-02,7128,4079
2,2023-03,7737,4430
3,2023-04,7425,4237
4,2023-05,7729,4446


# 5.7 Price vs quantity

In [18]:
price_quantity_corr = sales_data[
    ['quantity', 'unit_price_order']
].corr()

price_quantity_corr

,quantity,unit_price_order
quantity,1.000000,-0.005333
unit_price_order,-0.005333,1.000000


# 5.8 Discount vs quantity

In [19]:
discount_quantity_corr = sales_data[
    ['quantity', 'discount_pct']
].corr()

discount_quantity_corr

,quantity,discount_pct
quantity,1.000000,0.000393
discount_pct,0.000393,1.000000


Demand Analysis is an EDA step performed before ML. It helps us understand historical demand patterns such as:
- Total quantity sold → overall demand
- Total orders → number of orders
- Category demand → which categories sell the most
- Product demand → which products are most popular
- Monthly demand → how demand changes over time
- Price vs quantity → relationship between price and demand
- Discount vs quantity → relationship between discounts and demand
The main purpose is to identify patterns and important factors that can be used during feature engineering and ML modeling.
In simple words: First understand the demand, then create features from those patterns, and finally train the ML model to predict future demand.

# 6. Feature Engineering
This is the most important stage before ML.
Instead of using individual order rows, we will create a monthly product-level dataset.

In [20]:
monthly_product_demand = (
    sales_data
    .groupby([
        'product_id',
        'order_month'
    ])
    .agg(
        monthly_quantity=('quantity', 'sum'),
        monthly_orders=('order_id', 'nunique'),
        average_discount=('discount_pct', 'mean'),
        average_price=('unit_price_order', 'mean')
    )
    .reset_index()
)
monthly_product_demand.head()

,product_id,order_month,monthly_quantity,monthly_orders,average_discount,average_price
0,PROD00001,2023-01,1,1,0.000000,446.04
1,PROD00001,2023-02,5,3,13.333333,446.04
2,PROD00001,2023-09,2,2,2.500000,446.04
3,PROD00001,2023-11,1,1,25.000000,446.04
4,PROD00001,2023-12,2,2,15.000000,446.04


## 6.2 Previous Month Demand

In [28]:
monthly_product_demand = (
    monthly_product_demand
    .sort_values([
        'product_id',
        'order_month'
    ])
)


In [29]:
monthly_product_demand[
    'previous_month_quantity'
] = (
    monthly_product_demand
    .groupby('product_id')['monthly_quantity']
    .shift(1)
)
monthly_product_demand.head()

,product_id,order_month,monthly_quantity,monthly_orders,average_discount,average_price,previous_month_quantity
0,PROD00001,2023-01,1,1,0.000000,446.04,NaN
1,PROD00001,2023-02,5,3,13.333333,446.04,1.0
2,PROD00001,2023-09,2,2,2.500000,446.04,5.0
3,PROD00001,2023-11,1,1,25.000000,446.04,2.0
4,PROD00001,2023-12,2,2,15.000000,446.04,1.0


## 6.3 Previous 2 Month Demand

In [30]:
monthly_product_demand[
    'previous_2_month_quantity'
] = (
    monthly_product_demand
    .groupby('product_id')['monthly_quantity']
    .shift(2)
)


In [32]:
monthly_product_demand.head()

,product_id,order_month,monthly_quantity,monthly_orders,average_discount,average_price,previous_month_quantity,previous_2_month_quantity
0,PROD00001,2023-01,1,1,0.000000,446.04,NaN,NaN
1,PROD00001,2023-02,5,3,13.333333,446.04,1.0,NaN
2,PROD00001,2023-09,2,2,2.500000,446.04,5.0,1.0
3,PROD00001,2023-11,1,1,25.000000,446.04,2.0,5.0
4,PROD00001,2023-12,2,2,15.000000,446.04,1.0,2.0


## 6.4 Previous 3 Month Demand

In [35]:
monthly_product_demand[
    'previous_3_month_quantity'
] = (
    monthly_product_demand
    .groupby('product_id')['monthly_quantity']
    .shift(3)
)
monthly_product_demand.head()

,product_id,order_month,monthly_quantity,monthly_orders,average_discount,average_price,previous_month_quantity,previous_2_month_quantity,previous_3_month_quantity
0,PROD00001,2023-01,1,1,0.000000,446.04,NaN,NaN,NaN
1,PROD00001,2023-02,5,3,13.333333,446.04,1.0,NaN,NaN
2,PROD00001,2023-09,2,2,2.500000,446.04,5.0,1.0,NaN
3,PROD00001,2023-11,1,1,25.000000,446.04,2.0,5.0,1.0
4,PROD00001,2023-12,2,2,15.000000,446.04,1.0,2.0,5.0


These are called lag features.
Why?
Because historical demand is likely to contain useful information about future demand.

### 6.5 Add Product Characteristics
Now merge the product information:

In [37]:
monthly_product_demand = (
    monthly_product_demand.merge(
        products[
            [
                'product_id',
                'category',
                'sub_category',
                'brand',
                'unit_price',
                'stock_quantity',
                'rating'
            ]
        ],
        on='product_id',
        how='left'
    )
)
monthly_product_demand.head()

,product_id,order_month,monthly_quantity,monthly_orders,average_discount,average_price,previous_month_quantity,previous_2_month_quantity,previous_3_month_quantity,category_x,...,brand_x,unit_price_x,stock_quantity_x,rating_x,category_y,sub_category_y,brand_y,unit_price_y,stock_quantity_y,rating_y
0,PROD00001,2023-01,1,1,0.000000,446.04,NaN,NaN,NaN,Sports,...,Orion,446.04,266,4.5,Sports,Fitness Equipment,Orion,446.04,266,4.5
1,PROD00001,2023-02,5,3,13.333333,446.04,1.0,NaN,NaN,Sports,...,Orion,446.04,266,4.5,Sports,Fitness Equipment,Orion,446.04,266,4.5
2,PROD00001,2023-09,2,2,2.500000,446.04,5.0,1.0,NaN,Sports,...,Orion,446.04,266,4.5,Sports,Fitness Equipment,Orion,446.04,266,4.5
3,PROD00001,2023-11,1,1,25.000000,446.04,2.0,5.0,1.0,Sports,...,Orion,446.04,266,4.5,Sports,Fitness Equipment,Orion,446.04,266,4.5
4,PROD00001,2023-12,2,2,15.000000,446.04,1.0,2.0,5.0,Sports,...,Orion,446.04,266,4.5,Sports,Fitness Equipment,Orion,446.04,266,4.5


### 6.6 Remove rows without historical demand
The first few observations for each product won't have previous 1, 2, or 3 months. 
so

In [38]:
ml_data = monthly_product_demand.dropna(
    subset=[
        'previous_month_quantity',
        'previous_2_month_quantity',
        'previous_3_month_quantity'
    ]
).copy()

In [39]:
ml_data.shape

(96521, 21)

- 6.1: Converts order-level data into a product-month level dataset, which gives the ML model the correct structure for demand prediction.

- 6.2: Adds previous month demand so the model can use recent demand to predict future demand.

- 6.3: Adds previous 2-month demand to capture medium-term demand patterns.

- 6.4: Adds previous 3-month demand to capture longer-term demand patterns.

- 6.5: Adds price, category, brand, rating, discount, and stock information so the model can learn factors that may influence demand.

- 6.6: Removes rows without enough historical demand so the ML model trains only on complete and reliable data.

# 7. Train/Test Split
Because we're predicting future demand, we should not randomly split the data.
- Instead:
- 2023–2024 → Training
- 2025  → Testing

### 7.1 Training data

In [47]:
train_data = ml_data[
    ml_data['order_month'] < '2025-01'
].copy()
print(train_data.columns.tolist())

['product_id', 'order_month', 'monthly_quantity', 'monthly_orders', 'average_discount', 'average_price', 'previous_month_quantity', 'previous_2_month_quantity', 'previous_3_month_quantity', 'category_x', 'sub_category_x', 'brand_x', 'unit_price_x', 'stock_quantity_x', 'rating_x', 'category_y', 'sub_category_y', 'brand_y', 'unit_price_y', 'stock_quantity_y', 'rating_y']


### 7.2 Testing data

In [42]:
test_data = ml_data[
    ml_data['order_month'] >= '2025-01'
].copy()

In [43]:
print("Training:", train_data.shape)
print("Testing:", test_data.shape)

Training: (54496, 21)
Testing: (42025, 21)


## Why time-based splitting?
Because in a real business situation:
We use the past to predict the future.

We should never allow future data to influence the training process.

# 8. Multiple Linear Regression
Now we actually build the ML model.

### 8.1 Define features

In [49]:
features = [
    'previous_month_quantity',
    'previous_2_month_quantity',
    'previous_3_month_quantity',
    'average_discount',
    'average_price',
    'unit_price_x',
    'stock_quantity_x',
    'rating_x',
    'category_x',
    'sub_category_x',
    'brand_x'
]

## 8.2 Define X and Y

In [50]:
#Training
X_train = train_data[features]

y_train = train_data['monthly_quantity']

In [52]:
# Testing
X_test = test_data[features]

y_test = test_data[
    'monthly_quantity'
]

## 8.3 Separate categorical and numerical variables

In [66]:
categorical_features = [
    'category_x',
    'sub_category_x',
    'brand_x'
]

In [68]:
numerical_features = [
    'previous_month_quantity',
    'previous_2_month_quantity',
    'previous_3_month_quantity',
    'average_discount',
    'average_price',
    'unit_price_x',
    'stock_quantity_x',
    'rating_x'
]

### 8.4 One-Hot Encoding
Linear Regression cannot directly understand:
- Sports
- Books
- Fashion
- Electronics
So we convert categorical variables into numerical columns.

In [62]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

In [69]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            'passthrough',
            numerical_features
        ),
        (
            'cat',
            OneHotEncoder(handle_unknown='ignore'),
            categorical_features
        )
    ]
)

## 8.5 Create the Linear Regression Pipeline

In [70]:
model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('regressor', LinearRegression())
    ]
)

## 8.6 Train the model

In [71]:
model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [73]:
y_pred = model.predict(X_test)

In [74]:
# Create a prediction table:
prediction_result = pd.DataFrame({
    'Actual_Demand': y_test.values,
    'Predicted_Demand': y_pred
})

In [75]:
prediction_result.head(10)

,Actual_Demand,Predicted_Demand
0,4,2.149641
1,2,2.173044
2,1,2.159540
3,1,2.148086
4,2,2.163200
5,3,2.167831
6,5,2.193293
7,2,2.199801
8,1,2.144317
9,1,2.140705


In [77]:
prediction_result['Error'] = (
    prediction_result['Actual_Demand']
    - prediction_result['Predicted_Demand']
)
prediction_result

,Actual_Demand,Predicted_Demand,Error
0,4,2.149641,1.850359
1,2,2.173044,-0.173044
2,1,2.159540,-1.159540
3,1,2.148086,-1.148086
4,2,2.163200,-0.163200
...,...,...,...
42020,3,2.072385,0.927615
42021,1,2.098595,-1.098595
42022,1,2.076142,-1.076142
42023,4,2.075395,1.924605


# 10. R²
R² tells us how well the model explains variation in product demand.

In [78]:
from sklearn.metrics import r2_score

r2 = r2_score(
    y_test,
    y_pred
)

print("R² Score:", r2)

R² Score: -0.00052367028134781


R² measures how well your model explains the variation in product demand.
- R² = 1 → Excellent predictions
- R² = 0 → Model explains almost none of the variation
- R² < 0 → Model performs worse than simply predicting the average demand

My Linear Regression failed because demand has complex, nonlinear patterns that a simple linear model could not capture. The model also lacked strong time/seasonality features, and the train/test setup may not have been ideal for time-based demand prediction.
R² = -0.00052 → the model explains almost none of the demand variation and performs slightly worse than a simple average-demand prediction.